In [2]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf

In [3]:
# Excel-Datei laden (fertig gemergte Tabelle)
data = pd.read_excel(
    "Daten/merged_data.xlsx",
    sheet_name=0  # erstes (und einziges) Blatt
)

# Überblick
data.head()

,Jahr,Gruppe,Unfälle_total,Unfälle_leicht,Unfälle_schwer,Unfälle_tödlich,Bevölkerung
0,2015,0-19,617,482,130,5,1674330.0
1,2015,20-90+,3014,2027,930,57,6655670.0
2,2016,0-19,595,464,126,5,1692420.0
3,2016,20-90+,2742,1870,835,37,6727580.0
4,2017,0-19,775,636,134,5,1696000.0


In [4]:
data.columns

Index(['Jahr', 'Gruppe', 'Unfälle_total', 'Unfälle_leicht', 'Unfälle_schwer',
       'Unfälle_tödlich', 'Bevölkerung'],
      dtype='object')

In [5]:
# Jahr numerisch
data["Jahr"] = data["Jahr"].astype(int)

# Altersgruppe binär codieren
data["jung"] = (data["Gruppe"] == "0-19").astype(int)

# Offset vorbereiten
data["log_bev"] = np.log(data["Bevölkerung"])

# Finale Kontrolle
data[["Jahr", "Gruppe", "jung", "Unfälle_total", "Bevölkerung", "log_bev"]].head()

,Jahr,Gruppe,jung,Unfälle_total,Bevölkerung,log_bev
0,2015,0-19,1,617,1674330.0,14.330924
1,2015,20-90+,0,3014,6655670.0,15.710980
2,2016,0-19,1,595,1692420.0,14.341670
3,2016,20-90+,0,2742,6727580.0,15.721726
4,2017,0-19,1,775,1696000.0,14.343783


In [6]:
data.describe()

,Jahr,Unfälle_total,Unfälle_leicht,Unfälle_schwer,Unfälle_tödlich,Bevölkerung,jung,log_bev
count,20.000000,20.000000,20.000000,20.000000,20.000000,2.000000e+01,20.000000,20.000000
mean,2019.500000,1771.600000,1240.900000,507.850000,22.850000,4.331000e+06,0.500000,15.057461
std,2.946898,1065.420125,694.400832,354.577625,18.296606,2.671425e+06,0.512989,0.712245
min,2015.000000,499.000000,377.000000,119.000000,1.000000,1.674330e+06,0.000000,14.330924
25%,2017.000000,735.500000,572.500000,133.000000,5.000000,1.724498e+06,0.000000,14.360446
50%,2019.500000,1846.500000,1309.500000,523.000000,18.000000,4.228310e+06,0.500000,15.057402
75%,2022.000000,2783.000000,1915.000000,837.250000,39.500000,6.902168e+06,1.000000,15.747340
max,2024.000000,3014.000000,2027.000000,930.000000,57.000000,7.249050e+06,1.000000,15.796381


Die Analyse basiert auf aggregierten Jahresdaten (n = 20). Aufgrund der kleinen Stichprobe liegt der Fokus auf Effektgrössen und Trends, nicht auf hochpräziser Inferenz.

In [7]:
model_base = smf.glm(
    formula="Unfälle_total ~ jung + Jahr",
    data=data,
    family=sm.families.Poisson(),
    offset=data["log_bev"]
).fit()

model_base.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:          Unfälle_total   No. Observations:                   20
Model:                            GLM   Df Residuals:                       17
Model Family:                 Poisson   Df Model:                            2
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -378.16
Date:                Sat, 17 Jan 2026   Deviance:                       574.41
Time:                        13:19:15   Pearson chi2:                     582.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.7762
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.3868      3.732     -0.104      0.917      -7.701       6.928
jung           0.0668      0.013      5.127      0.000       0.041       0.092
Jahr          -0.0037      0.002     -1.991      0.047      -0.007   -5.65e-05
==============================================================================
"""

Im Durchschnitt weisen junge Personen eine signifikant höhere Unfallrate auf. Über alle Altersgruppen hinweg zeigt sich jedoch ein leicht rückläufiger Trend über die Zeit.


In [8]:
model_inter = smf.glm(
    formula="Unfälle_total ~ jung * Jahr",
    data=data,
    family=sm.families.Poisson(),
    offset=data["log_bev"]
).fit()

model_inter.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:          Unfälle_total   No. Observations:                   20
Model:                            GLM   Df Residuals:                       16
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -207.14
Date:                Sat, 17 Jan 2026   Deviance:                       232.37
Time:                        13:19:15   Pearson chi2:                     223.
No. Iterations:                     5   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     35.2373      4.202      8.385      0.000      27.001      43.474
jung        -169.9898      9.240    -18.396      0.000    -188.101    -151.879
Jahr          -0.0213      0.002    -10.244      0.000      -0.025      -0.017
jung:Jahr      0.0842      0.005     18.406      0.000       0.075       0.093
==============================================================================
"""

Die zeitliche Entwicklung der Unfallraten unterscheidet sich signifikant zwischen jungen und älteren Personen. Während die Unfallrate bei Älteren sinkt, steigt sie bei Jungen deutlich an.


In [9]:
print(model_base.aic, model_inter.aic)

762.3260988422744 422.287000013137


In [10]:
for var in ["Unfälle_leicht", "Unfälle_schwer", "Unfälle_tödlich"]:
    print(f"\n### {var}")
    m = smf.glm(
        formula=f"{var} ~ jung * Jahr",
        data=data,
        family=sm.families.Poisson(),
        offset=data["log_bev"]
    ).fit()
    print(m.summary().tables[1])


### Unfälle_leicht
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     33.7467      5.089      6.631      0.000      23.772      43.721
jung        -153.2280     10.649    -14.389      0.000    -174.099    -132.357
Jahr          -0.0208      0.003     -8.242      0.000      -0.026      -0.016
jung:Jahr      0.0760      0.005     14.408      0.000       0.066       0.086

### Unfälle_schwer
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     34.4903      7.622      4.525      0.000      19.551      49.430
jung        -220.8914     19.229    -11.487      0.000    -258.580    -183.203
Jahr          -0.0215      0.004     -5.706      0.000      -0.029      -0.014
jung:Jahr      0.1092      0.010     11.476      0.000       0.091       0.128

### Unfälle

Leichte Unfälle: Der Anstieg der Unfälle bei Jugendlichen wird primär durch leichte Unfälle getrieben. Bei älteren Personen nehmen leichte Unfälle hingegen über die Zeit ab.
Schwere Unfälle: Auch schwere Unfälle zeigen bei Jugendlichen einen zunehmenden Trend, während sie bei Älteren rückläufig sind.
Tödliche Unfälle: Für tödliche Unfälle deutet sich ebenfalls ein positiver Trend bei Jugendlichen an, jedoch ist die Unsicherheit aufgrund der sehr geringen Ereigniszahlen hoch.

Die Poisson-Regression mit Offset zeigt, dass sich die zeitlichen Trends der Unfallraten signifikant zwischen jungen und älteren Personen unterscheiden. Während die Unfallraten bei älteren Personen über die Zeit abnehmen, nehmen sie bei jungen Personen deutlich zu, insbesondere bei leichten und schweren Unfällen. Aufgrund der begrenzten Anzahl an Beobachtungen (n=20) und fehlender Expositionsdaten (z. B. Anzahl Motorradfahrer*innen) sind die Resultate vorsichtig zu interpretieren, deuten jedoch auf einen strukturellen Unterschied im Unfallgeschehen hin.


Interpretation:
Zur Untersuchung der zeitlichen Entwicklung der Unfallraten wurde eine Poisson-Regression mit Log-Link geschätzt, wobei der Logarithmus der Bevölkerung als Offset verwendet wurde. Damit wird die Unfallrate pro Person modelliert.

Im Basismodell ohne Interaktion zeigt sich, dass junge Personen eine signifikant höhere Unfallrate aufweisen als die ältere Bevölkerung (Koeffizient jung = 0.0668, p < 0.001). Zudem deutet der negative Jahreseffekt (−0.0037, p = 0.047) auf einen leichten allgemeinen Rückgang der Unfallrate über die Zeit hin. Dieses Modell erklärt einen grossen Teil der Variation (Pseudo-R² = 0.776), erfasst jedoch keine altersabhängigen Zeittrends.

Das Interaktionsmodell mit jung × Jahr verbessert die Modellanpassung deutlich (AIC 422 vs. 762). Der negative Haupteffekt von Jahr (−0.0213, p < 0.001) zeigt, dass die Unfallrate bei älteren Personen jährlich um rund 2 % sinkt. Der positive und hochsignifikante Interaktionseffekt (jung:Jahr = 0.0842, p < 0.001) weist hingegen darauf hin, dass die Unfallrate bei jungen Personen über die Zeit zunimmt. Insgesamt ergibt sich für junge Personen eine jährliche Zunahme der Unfallrate von rund 6 %.

Die getrennte Analyse nach Unfallschwere bestätigt dieses Muster: Bei leichten und schweren Unfällen steigen die Unfallraten bei jungen Personen signifikant an, während sie bei älteren Personen zurückgehen. Auch bei tödlichen Unfällen zeigt sich ein positiver, wenn auch mit grösserer Unsicherheit geschätzter Trend für junge Personen.

Zusammenfassend zeigen die Ergebnisse konsistent, dass sich die Unfallentwicklung seit 2015 stark nach Altersgruppen unterscheidet. Während die Unfallraten der älteren Bevölkerung deutlich sinken, nehmen sie bei jungen Personen zu. Diese Resultate sind statistisch robust, müssen jedoch aufgrund der kleinen Stichprobe und der fehlenden Expositionsdaten vorsichtig interpretiert werden, kausal nicht eindeutig interpretierbar.

In [14]:
# Post-2021 Dummy
data["post2021"] = (data["Jahr"] >= 2021).astype(int)

# Jahr zentrieren (Referenz = 2021)
data["Jahr_zentriert"] = data["Jahr"] - 2021

In [15]:
data["jung"] = (data["Gruppe"] == "0-19").astype(int)

In [16]:
data[["Jahr", "Gruppe", "jung", "post2021", "Jahr_zentriert"]].head(6)

,Jahr,Gruppe,jung,post2021,Jahr_zentriert
0,2015,0-19,1,0,-6
1,2015,20-90+,0,0,-6
2,2016,0-19,1,0,-5
3,2016,20-90+,0,0,-5
4,2017,0-19,1,0,-4
5,2017,20-90+,0,0,-4


In [17]:
glm_h1 = smf.glm(
    formula="Unfälle_total ~ post2021 + Jahr_zentriert",
    data=data[data["jung"] == 1],
    family=sm.families.Poisson(),
    offset=np.log(data[data["jung"] == 1]["Bevölkerung"])
).fit()

print(glm_h1.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:          Unfälle_total   No. Observations:                   10
Model:                            GLM   Df Residuals:                        7
Model Family:                 Poisson   Df Model:                            2
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -104.16
Date:                Sat, 17 Jan 2026   Deviance:                       124.13
Time:                        13:24:23   Pearson chi2:                     126.
No. Iterations:                     4   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -7.9515      0.033   -243.

Zur Überprüfung der Hypothese H1 wurde für die Altersgruppe der Jugendlichen (0–19 Jahre) ein generalisiertes lineares Modell mit Poisson-Verteilung und Log-Link geschätzt. Die abhängige Variable sind die jährlichen Gesamtunfälle, wobei mittels eines logarithmischen Offsets die jeweilige Bevölkerungsgrösse berücksichtigt wird. Dadurch wird nicht die absolute Anzahl an Unfällen, sondern die Unfallrate modelliert. Als erklärende Variablen wurden ein Dummy für die Zeit nach 2021 sowie ein zentriertes Jahresmaß in das Modell aufgenommen, um sowohl einen möglichen strukturellen Bruch als auch einen linearen Zeittrend abzubilden.

Die Schätzung zeigt einen positiven und hochsignifikanten Effekt der Variable post2021. Der entsprechende Koeffizient von 0.433 impliziert, dass die Unfallrate bei Jugendlichen nach 2021 deutlich höher liegt als in den Jahren davor. In exponentiierter Form entspricht dies einem Anstieg der Unfallrate um rund 54 Prozent, bei ansonsten gleichen Bedingungen. Der Koeffizient des zentrierten Jahres ist hingegen klein und statistisch nicht signifikant, was darauf hindeutet, dass kein zusätzlicher linearer Trend über die gesamte Zeitperiode hinweg besteht, sobald zwischen der Zeit vor und nach 2021 unterschieden wird. Der beobachtete Anstieg ist somit primär auf einen Niveauwechsel nach 2021 zurückzuführen und nicht auf eine kontinuierliche Entwicklung über mehrere Jahre.

Inhaltlich belegt dieses Ergebnis einen klaren und statistisch robusten Anstieg der Unfallrate bei Jugendlichen nach 2021 und bestätigt damit Hypothese H1. Gleichzeitig erlaubt das Modell keine Aussagen über die Ursachen dieses Anstiegs, da es sich um eine rein deskriptiv-statistische Analyse auf aggregierter Ebene handelt. Zudem lassen sich aus diesem Modell weder Vergleiche mit anderen Altersgruppen noch differenzierte Aussagen zur Schwere der Unfälle ableiten, da ausschliesslich die Gesamtzahl der Unfälle innerhalb der Jugendpopulation betrachtet wird.

In [18]:
glm_h2 = smf.glm(
    formula="Unfälle_schwer ~ post2021 + Jahr_zentriert",
    data=data[data["jung"] == 1],
    family=sm.families.Poisson(),
    offset=np.log(data[data["jung"] == 1]["Bevölkerung"])
).fit()

print(glm_h2.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:         Unfälle_schwer   No. Observations:                   10
Model:                            GLM   Df Residuals:                        7
Model Family:                 Poisson   Df Model:                            2
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -41.816
Date:                Sat, 17 Jan 2026   Deviance:                       14.621
Time:                        13:24:36   Pearson chi2:                     14.7
No. Iterations:                     4   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -9.4674      0.070   -135.

Zur Überprüfung der Hypothese H2 wurde für die Altersgruppe der Jugendlichen (0–19 Jahre) ein generalisiertes lineares Modell mit Poisson-Verteilung und Log-Link geschätzt, bei dem die jährliche Anzahl schwerer Unfälle als abhängige Variable verwendet wird. Durch die Einbindung eines logarithmischen Offsets für die jeweilige Bevölkerungsgrösse wird die Unfallrate modelliert. Als erklärende Variablen wurden erneut ein Dummy für die Zeit nach 2021 sowie ein zentriertes Jahresmass berücksichtigt, um zwischen einem möglichen strukturellen Niveauwechsel und einem linearen Zeittrend zu unterscheiden.

Die Schätzergebnisse zeigen einen positiven und hochsignifikanten Effekt der Variable post2021. Der entsprechende Koeffizient von 0.479 impliziert, dass die Rate schwerer Unfälle bei Jugendlichen nach 2021 deutlich höher liegt als in den Jahren davor. In exponentiierter Form entspricht dies einem Anstieg der Rate schwerer Unfälle um rund 61 Prozent. Der Koeffizient des zentrierten Jahres ist hingegen statistisch nicht signifikant, was darauf hindeutet, dass kein zusätzlicher linearer Zeittrend über die gesamte Beobachtungsperiode hinweg besteht, sobald der Zeitraum nach 2021 explizit berücksichtigt wird. Auch hier deutet das Modell somit auf einen diskreten Niveauanstieg nach 2021 hin, nicht auf eine graduelle zeitliche Entwicklung.

Inhaltlich belegt dieses Ergebnis einen signifikanten Anstieg der Rate schwerer Unfälle bei Jugendlichen nach 2021 und bestätigt damit Hypothese H2. Gleichzeitig erlaubt das Modell keine Aussagen über die Ursachen dieses Anstiegs oder über individuelle Unfallrisiken, da die Analyse auf aggregierten Jahresdaten basiert. Zudem lassen sich aus diesem Modell keine Vergleiche mit anderen Altersgruppen ableiten, da ausschliesslich die Jugendpopulation betrachtet wird.

In [19]:
glm_h3 = smf.glm(
    formula="Unfälle_total ~ jung * Jahr_zentriert",
    data=data,
    family=sm.families.Poisson(),
    offset=np.log(data["Bevölkerung"])
).fit()

print(glm_h3.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:          Unfälle_total   No. Observations:                   20
Model:                            GLM   Df Residuals:                       16
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -207.14
Date:                Sat, 17 Jan 2026   Deviance:                       232.37
Time:                        13:25:14   Pearson chi2:                     223.
No. Iterations:                     5   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -7.8476    

Zur Überprüfung der Hypothese H3 wurde ein generalisiertes lineares Modell mit Poisson-Verteilung und Log-Link geschätzt, das sowohl Jugendliche (0–19 Jahre) als auch Erwachsene (20–90+ Jahre) umfasst. Die abhängige Variable ist die jährliche Anzahl der Gesamtunfälle, wobei mittels eines logarithmischen Offsets die jeweilige Bevölkerungsgrösse berücksichtigt wird, sodass Unfallraten modelliert werden. Als erklärende Variablen wurden ein Dummy für die Altersgruppe der Jugendlichen, ein zentriertes Jahresmass sowie deren Interaktion in das Modell aufgenommen, um altersgruppenspezifische zeitliche Entwicklungen der Unfallrate zu analysieren.

Die Schätzergebnisse zeigen zunächst einen signifikanten Haupteffekt der Variable jung, was darauf hindeutet, dass die Unfallrate bei Jugendlichen im Referenzjahr höher liegt als bei Erwachsenen. Der Koeffizient des zentrierten Jahres ist negativ und hochsignifikant, was auf einen rückläufigen Trend der Unfallrate bei Erwachsenen über die Zeit hinweist. Von zentraler Bedeutung für Hypothese H3 ist jedoch der Interaktionsterm zwischen jung und Jahr_zentriert, der positiv und hochsignifikant ausfällt. Dieser Befund impliziert, dass sich die Unfallrate bei Jugendlichen über die Zeit deutlich stärker erhöht als bei Erwachsenen.

Inhaltlich bedeutet dies, dass während bei Erwachsenen ein abnehmender oder stagnierender Unfalltrend beobachtet wird, sich die Unfallrate bei Jugendlichen im Zeitverlauf signifikant ungünstiger entwickelt. Damit wird Hypothese H3 klar bestätigt. Das Modell erlaubt jedoch keine kausalen Schlussfolgerungen über die Gründe dieser unterschiedlichen Entwicklung, da es sich um eine aggregierte Analyse auf Jahresebene handelt und mögliche externe Einflussfaktoren nicht explizit berücksichtigt werden.